# T13 — log-HAR et log-HAR-X (benchmarks statistiques supplémentaires)

**Pourquoi ce notebook.** Le HAR de `04_har` est estimé par MCO sur les niveaux de variance : il minimise le MSE. LightGBM est entraîné avec `objective="gamma"`, dont la déviance vaut 2 × QLIKE, la mesure qui sert à le juger. Pour une comparaison équitable, on ajoute deux benchmarks linéaires estimés sur les **logarithmes** :

- **LOGHAR** : ln y = β₀ + β_d ln rv_d + β_w ln rv_w + β_m ln rv_m + ε
- **LOGHARX** : la même chose + β_v ln VIX

**Intuition.** En log, une erreur relative (prévoir 2 au lieu de 1, ou 0,5 au lieu de 1) pèse pareil, comme dans la QLIKE. Le log rend aussi la distribution de rv beaucoup moins asymétrique, réduit le poids des pics (2008, 2020) dans l'estimation et garantit une prévision positive. Les coefficients deviennent des élasticités. Le VIX est une volatilité et rv une variance : en log, cette différence d'unité devient un simple facteur d'échelle (ln VIX² = 2 ln VIX), absorbé par le coefficient.

**Retour aux niveaux.** Si ε suit une loi normale de variance σ², alors E[y] = exp(μ + σ²/2). On multiplie donc exp(prévision log) par exp(σ̂²/2), avec σ̂² la variance résiduelle de l'entraînement. Sans cette correction, la prévision serait la médiane et non la moyenne, donc biaisée vers le bas.

**Protocole.** Identique au HAR : estimation initiale 2000–2014, test du 2015-01-02 au 2026-09-10 (2026-09-09 pour le TSX), ré-estimation tous les 22 jours, seules les cibles déjà observées à la date de prévision servent à l'estimation (`iloc[:-h]`).

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm

here = Path.cwd().resolve()
root = next(p for p in [here, *here.parents] if (p / "data" / "features_spx.csv").exists())
sys.path.append(str(root))
from src.utils import TEST_START, HORIZONS, save_forecasts, qlike, mse

FEAT = {k: pd.read_csv(root / "data" / f"features_{k}.csv", index_col=0, parse_dates=True)
        for k in ("spx", "tsx")}
SPECS = {"LOGHAR": ["rv_d", "rv_w", "rv_m"],
         "LOGHARX": ["rv_d", "rv_w", "rv_m", "vix"]}

# le log exige des valeurs strictement positives
for k, f in FEAT.items():
    assert (f[["rv_d", "rv_w", "rv_m", "vix", "y_h1", "y_h5", "y_h22"]] > 0).all().all(), k
    print(k, f.shape, f.index.min().date(), "→", f.index.max().date())

spx (6690, 10) 2000-02-03 → 2026-09-10
tsx (6670, 10) 2000-02-03 → 2026-09-09


## 1. Estimation sur 2000–2014 (écarts-types HAC)

In [2]:
rows = []
for ix, feat in FEAT.items():
    train = feat[feat.index < TEST_START]
    for name, cols in SPECS.items():
        X = sm.add_constant(np.log(train[cols]))
        for h in HORIZONS:
            # on retire les dernières lignes dont la cible déborde sur 2015
            Xh, yh = X.iloc[:-h], np.log(train[f"y_h{h}"]).iloc[:-h]
            res = sm.OLS(yh, Xh).fit(cov_type="HAC", cov_kwds={"maxlags": h})
            row = {"indice": ix, "modèle": name, "horizon": h}
            for v in Xh.columns:
                row[f"coef_{v}"] = res.params[v]
                row[f"se_{v}"] = res.bse[v]
            row["somme_betas_rv"] = res.params[["rv_d", "rv_w", "rv_m"]].sum()
            row["sigma2_resid"] = res.mse_resid
            row["R2_log"] = res.rsquared
            row["n_train"] = int(res.nobs)
            rows.append(row)

table2b = pd.DataFrame(rows).round(4)
table2b.to_csv(root / "results" / "table2b_loghar.csv", index=False)
display(table2b)

,indice,modèle,horizon,coef_const,se_const,coef_rv_d,se_rv_d,coef_rv_w,se_rv_w,coef_rv_m,se_rv_m,somme_betas_rv,sigma2_resid,R2_log,n_train,coef_vix,se_vix
0,spx,LOGHAR,1,-0.2485,0.0159,0.1185,0.0224,0.4300,0.0417,0.3822,0.0350,0.9307,0.6356,0.5352,3750,NaN,NaN
1,spx,LOGHAR,5,-0.0788,0.0202,0.1178,0.0160,0.3613,0.0439,0.4124,0.0425,0.8916,0.2829,0.7014,3746,NaN,NaN
2,spx,LOGHAR,22,-0.0621,0.0403,0.0773,0.0111,0.3026,0.0481,0.4258,0.0511,0.8058,0.2677,0.6663,3729,NaN,NaN
3,spx,LOGHARX,1,-5.3000,0.2911,0.0603,0.0212,0.2893,0.0402,-0.0271,0.0397,0.3224,0.5881,0.5701,3750,1.5871,0.0913
4,spx,LOGHARX,5,-4.3199,0.3293,0.0689,0.0148,0.2418,0.0418,0.0700,0.0485,0.3808,0.2494,0.7369,3746,1.3325,0.1039
5,spx,LOGHARX,22,-3.5791,0.5305,0.0379,0.0103,0.2039,0.0483,0.1394,0.0704,0.3812,0.2447,0.6950,3729,1.1051,0.1657
6,tsx,LOGHAR,1,-0.3024,0.0198,0.1245,0.0193,0.4329,0.0353,0.3403,0.0315,0.8978,0.6858,0.4969,3736,NaN,NaN
7,tsx,LOGHAR,5,-0.1143,0.0258,0.0996,0.0130,0.4095,0.0371,0.3622,0.0360,0.8712,0.3089,0.6718,3732,NaN,NaN
8,tsx,LOGHAR,22,-0.0946,0.0492,0.0800,0.0101,0.2705,0.0504,0.4484,0.0493,0.7988,0.2902,0.6418,3715,NaN,NaN
9,tsx,LOGHARX,1,-1.5185,0.1693,0.1144,0.0191,0.4064,0.0350,0.2606,0.0327,0.7814,0.6769,0.5036,3736,0.3804,0.0528


Lecture : un coefficient est une élasticité. Par exemple, un β_w de 0,4 signifie qu'une variance de la semaine 10 % plus élevée augmente la variance prévue d'environ 4 %. Le R² est calculé sur les logs : il n'est pas comparable à celui du HAR en niveaux.

## 2. Prévisions walk-forward

In [3]:
def log_har_walk_forward(feat, h, name, refit=22):
    cols = SPECS[name]
    # mêmes dates de test que les autres modèles
    dates = feat.index[(feat.index >= TEST_START) & feat["y_h22"].notna()]
    out, res, s2 = [], None, None
    for k, d in enumerate(dates):
        if k % refit == 0:                          # ré-estimation tous les 22 jours
            train = feat.loc[:d].iloc[:-h]          # seulement des cibles déjà connues à d
            res = sm.OLS(np.log(train[f"y_h{h}"]),
                         sm.add_constant(np.log(train[cols]))).fit()
            s2 = res.mse_resid                      # variance résiduelle, pour la correction
        x = sm.add_constant(np.log(feat.loc[[d], cols]), has_constant="add")
        f = float(np.exp(res.predict(x).iloc[0] + s2 / 2))   # E[y] = exp(mu + sigma²/2)
        out.append({"date": d, "model": name, "horizon": h,
                    "forecast": max(f, 1e-4), "actual": feat.loc[d, f"y_h{h}"]})
    return pd.DataFrame(out)

for ix, feat in FEAT.items():
    for name in SPECS:
        df = pd.concat([log_har_walk_forward(feat, h, name) for h in HORIZONS], ignore_index=True)
        save_forecasts(df, name, ix)                # results/forecasts_<ix>_<name>.csv
        for h, g in df.groupby("horizon"):
            print(f"{ix} {name:8s} h = {h:2d}  QLIKE = {qlike(g.actual, g.forecast):.4f}"
                  f"  MSE = {mse(g.actual, g.forecast):.4f}  n = {len(g)}")

spx LOGHAR   h =  1  QLIKE = 0.3839  MSE = 1.3690  n = 2939
spx LOGHAR   h =  5  QLIKE = 0.2684  MSE = 1.0033  n = 2939
spx LOGHAR   h = 22  QLIKE = 0.3189  MSE = 0.8077  n = 2939


spx LOGHARX  h =  1  QLIKE = 0.3394  MSE = 1.0424  n = 2939
spx LOGHARX  h =  5  QLIKE = 0.2286  MSE = 0.8075  n = 2939
spx LOGHARX  h = 22  QLIKE = 0.2923  MSE = 0.7882  n = 2939


tsx LOGHAR   h =  1  QLIKE = 0.3146  MSE = 1.3482  n = 2933
tsx LOGHAR   h =  5  QLIKE = 0.2068  MSE = 1.0943  n = 2933
tsx LOGHAR   h = 22  QLIKE = 0.3754  MSE = 0.9106  n = 2933


tsx LOGHARX  h =  1  QLIKE = 0.3024  MSE = 1.2147  n = 2933
tsx LOGHARX  h =  5  QLIKE = 0.1952  MSE = 1.0197  n = 2933
tsx LOGHARX  h = 22  QLIKE = 0.3616  MSE = 0.9091  n = 2933


## 3. Robustesse : choix de la correction de biais

On compare, à h = 1, trois façons de revenir aux niveaux : aucune correction (médiane), exp(σ̂²/2) (retenue, hypothèse de normalité des résidus en log) et l'estimateur de Duan (moyenne de exp(résidus), sans hypothèse de loi). Ce test n'est pas sauvegardé : il sert seulement à vérifier que la conclusion ne dépend pas de ce choix.

In [4]:
def variante(feat, h, cols, mode, refit=22):
    dates = feat.index[(feat.index >= TEST_START) & feat["y_h22"].notna()]
    fc = []
    for k, d in enumerate(dates):
        if k % refit == 0:
            train = feat.loc[:d].iloc[:-h]
            res = sm.OLS(np.log(train[f"y_h{h}"]), sm.add_constant(np.log(train[cols]))).fit()
            adj = {"aucune": 1.0, "normale": np.exp(res.mse_resid / 2),
                   "Duan": np.mean(np.exp(res.resid))}[mode]
        x = sm.add_constant(np.log(feat.loc[[d], cols]), has_constant="add")
        fc.append(float(np.exp(res.predict(x).iloc[0])) * adj)
    return qlike(feat.loc[dates, f"y_h{h}"], fc)

rob = pd.DataFrame([{"indice": ix, "modèle": name,
                     **{m: variante(feat, 1, SPECS[name], m) for m in ("aucune", "normale", "Duan")}}
                    for ix, feat in FEAT.items() for name in SPECS]).round(4)
display(rob)

,indice,modèle,aucune,normale,Duan
0,spx,LOGHAR,0.4482,0.3839,0.3839
1,spx,LOGHARX,0.3708,0.3394,0.3397
2,tsx,LOGHAR,0.3603,0.3146,0.3150
3,tsx,LOGHARX,0.3349,0.3024,0.3030
